# NCF type algorithm implementation in EasyStudy and Novelty optimization
    
Two main tables: **participants** and **interactions** form EasyStudy

Additional **ratings** table added from original data for popularity computation

## Study details:
- evaluate two recommending algorithms; NCF = Neural Collaborative Filtering based algorithm, EASE = already implemented Embarrassingly Shallow Autoencoders from EasyStudy framework
- at each iteration (5 in total), results of both algorithms were displayed and organized into columns
    - at each iteration, users can select items they're interested in, rate individual algorithms (1-5 scale) and provide pairwise comparison of algorithms
- research included minor adjustment in NCF training function to improve Novelty, focus of evaluation should reflect that

## Implementation
- Implementation of the NCF algorithm can be found inside a forked repository from the original *EasyStudy* on separate branch called *simek_final_project*:
- https://github.com/SimekJan/EasyStudy/tree/simek_final_project

## Implementation commentary
- NCF algorithm was changed to get users recent history instead of user itself to correctly implement the AlgorithmBase interface which provides only selected_items as a parameter for predict method.
- Other than that the largest difference is in how negatives are chosen during training. We are choosing negatives for popular items more often to improve novelty among the results.
- DISCLAIMER: On my (not very well equipped) computer the training took very long time with a lot of struggle. For that reason I did not try to optimize hyperparameters which could have resulted in better overall results. 


In [1]:
import pandas as pd
import numpy as np
import os

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load popularity data

In [2]:
df_ratings = pd.read_csv("ratings.csv", index_col=0)

In [3]:
popularity = (
    df_ratings[df_ratings["rating"] > 2.5]
    .groupby("movieId")
    .size()
    .rename("popularity")
)

# Load data

In [5]:
df_interaction = pd.read_json("interaction.json", encoding='utf-8')
df_completed_participation = pd.read_csv("participation.csv", index_col=0)

# Enrich the interactions data frame

In [8]:
N_ITERATIONS = 8
def get_iteration(x):
    return json.loads(x)["iteration"]

In [9]:
import json
def set_iteration(row):
    if row.interaction_type == "iteration-started" or row.interaction_type == "iteration-ended":
        row['iteration'] = json.loads(row.data)['iteration']
    else:
        row['iteration'] = None
    return row

def set_result_layout(row):
    if row.interaction_type == "iteration-started":
        row['result_layout'] = json.loads(row.data)['result_layout']
    else:
        row['result_layout'] = None
    return row

def set_mapping(row):
    if row.interaction_type == 'iteration-started':
        dat = json.loads(row.data)['algorithm_assignment'].values()
        for mapping in dat:
            row[mapping['name'].upper()] = mapping['order']
    else:
        row['EASE'] = None
        row['NCF'] = None
    return row

d = df_interaction.copy()
d = d.set_index("id")
d = d.apply(set_iteration, axis=1).apply(set_result_layout, axis=1).apply(set_mapping, axis=1)
d['iteration'] = d.groupby('participation')['iteration'].ffill()
d['result_layout'] = d.groupby('participation')['result_layout'].ffill()
d['EASE'] = d.groupby('participation')['EASE'].ffill()
d['NCF'] = d.groupby('participation')['NCF'].ffill()
d = d[d.iteration.notna()]

# Filter only the information about item selections and iteration start

In [10]:
d["variant"] = -1

d.loc[d["interaction_type"] == "selected-item", "variant"] = (
    d.loc[d["interaction_type"] == "selected-item", "data"]
     .map(lambda x: json.loads(x)["selected_item"])
     .map(lambda x: x.get("variant", -1))
)

d = d.loc[d.iteration <= 8]

selected_item_interactions = d[d["variant"] >= 0].copy()

iteration_started_interactions = (
    d[d["interaction_type"] == "iteration-started"]
    .copy()
)

In [11]:
def getSelectedMovieId(x):
    return json.loads(x)["selected_item"]["movie_id"]

selected_item_interactions["movieID"] = np.nan
selected_item_interactions.movieID = selected_item_interactions.data.map(lambda x: getSelectedMovieId(x))

In [12]:
selected_item_interactions["selected_algorithm"] = "EASE"
selected_item_interactions.loc[selected_item_interactions.variant == selected_item_interactions.NCF, "selected_algorithm"] = "NCF"

## Result 1: Which algorithm was more successful in predicting users taste
- Determined by simple selected items count

In [26]:
algorithm_successfulness = (
    selected_item_interactions
    .groupby("selected_algorithm")["movieID"]
    .count()
)

### Quality results

In [27]:
print("EASE successfulness:", algorithm_successfulness["EASE"])
print("NCF successfulness:", algorithm_successfulness["NCF"])

EASE successfulness: 37
NCF successfulness: 29


## Result 2: Which algorithm suggested more niche items
- Determined by shown items novelty mean

In [17]:
ease_lists = []
ncf_lists = []

for row in iteration_started_interactions["data"]:
    data = json.loads(row)

    shown_obj = data.get("shown", None)
        
    if not isinstance(shown_obj, dict):
        continue

    ease = shown_obj.get("EASE", None)
    ncf = shown_obj.get("NCF", None)

    if ease:
        ease_lists.append(ease[-1])  # flatten [[...]]
    if ncf:
        ncf_lists.append(ncf[-1])

In [18]:
ease_items = [mid for lst in ease_lists for mid in lst]
ncf_items = [mid for lst in ncf_lists for mid in lst]

In [19]:
ease_pop = [popularity.get(mid, 0) for mid in ease_items]
ncf_pop = [popularity.get(mid, 0) for mid in ncf_items]

In [20]:
total_pop = popularity.sum()

ease_novelty = np.mean([
    -np.log2((p + 1) / (total_pop + len(popularity)))
    for p in ease_pop
])

ncf_novelty = np.mean([
    -np.log2((p + 1) / (total_pop + len(popularity)))
    for p in ncf_pop
])

### Novelty results

In [23]:
print("EASE novelty:", ease_novelty)
print("NCF novelty:", ncf_novelty)

EASE novelty: 16.337309713292505
NCF novelty: 16.70512477428268


## Final thoughts
- From the small number of collected data we cannot be sure that the negative looking results must be true. Further research needs to be done to confirm that.
- Furthermore if the model went through hyperparameter tuning we could have achieved much better results as well.